# 05. 목적지×월 수요 모델링·검증과 프로모션 검토

2026년 7월까지 관측된 노선 수요로 169개 목적지의 2026년 8–12월을 예측한다. B0, B1, Ridge와 트리 보정 모델을 2025년 Rolling Validation에서 비교한 다음, 선택한 모델만 2026년 Fixed Test에 적용한다. 마지막으로 같은 모델로 미래 수요와 검토 후보를 산출한다.

## 분석 목표와 평가 기간

목적지×월 여객 수요를 예측해 2026년 8–12월에 먼저 확인할 프로모션 후보를 좁힌다. 입력은 2023년 1월–2026년 7월의 월간 여객·운항 자료다. 24개월 이상 관측된 목적지만 모델 대상으로 사용하며, 대상은 169곳이다.

| 단계 | 예측 기준월 | 평가·예측 대상월 | 용도 |
|---|---|---|---|
| Rolling Validation | 2025년 1–7월 | 2025년 2–12월 | 모델·설정 선택 |
| Fixed Test | 2025년 8월–2026년 6월 | 2026년 1–7월 | 선택한 모델의 새 기간 성능 확인 |
| 미래 예측 | 2026년 7월 | 2026년 8–12월 | 수요·프로모션 검토 후보 산출 |

각 기준월에서 이미 알 수 있는 자료만 피처에 사용한다. 같은 대상월이 서로 다른 기준월에서 예측되므로 검증·Test의 평가 행에는 예측거리 h1–h5가 함께 있다. 지표는 실제 수요 합계를 분모로 한 **WAPE**와 전체 예측의 방향을 보여주는 **순편향**이다.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sqlalchemy import text
from IPython.display import display
ROOT=Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'scripts'))
import portfolio_data as pdx
import feature_layer as fl
engine=pdx.get_engine(); SQL=pdx.named_sql('05_demand_forecast.sql')
def query(name):return pdx.query(engine,SQL,name)
PLOTLY_TEMPLATE='plotly_white'
THREE_MONTH_HOLDOUT=ROOT/'data/legacy_holdout_2026.csv'
pdx.apply_ddl(engine,SQL,['create_v_model_destination_monthly_all','create_v_demand_destination_profile'])
print('모델 패널과 표본 뷰 준비 완료')

모델 패널과 표본 뷰 준비 완료


## 1. 예측 단위 결정

우선 3개월 합계 수요를 예측했을 때의 오차를 확인한다. 이 단위가 월별 프로모션 시기를 고르는 데 충분한지 판단한 뒤 월간 모델을 설계한다.

In [2]:
# 3개월 합계 수요 예측의 오차 비교
h = pd.read_csv(THREE_MONTH_HOLDOUT)
A = 'actual_passengers_next_3m'
MODELS = {'LGBM': 'predicted_passengers_next_3m', 'mean_naive': 'mean_naive_prediction',
          'seasonal_naive': 'seasonal_naive_prediction'}
def wape(df, pred, actual=A): return (df[pred] - df[actual]).abs().sum() / df[actual].sum()

three_month_overall = pd.Series({m: wape(h, c) for m, c in MODELS.items()}, name='holdout_WAPE')
display(three_month_overall.to_frame())

# 월별 시기를 판단할 수 있는지 함께 검토한다.


,holdout_WAPE
LGBM,0.122099
mean_naive,0.137964
seasonal_naive,0.158979


3개월 합계의 WAPE는 LGBM <strong>12.21%</strong>, 최근평균 13.80%, 전년동월 15.90%다. LGBM이 가장 낮지만 최근평균과 차이는 1.59%p다. 더구나 3개월 합계로는 8월과 10월 중 어느 달의 수요가 큰지 알 수 없다. 의사결정 단위에 맞춰 <strong>목적지×월 예측</strong>을 검증한다.

## 2. 모델 대상과 표본 기준

월별 관측이 24개월 이상인 목적지만 모델 대상에 포함한다. 기록이 너무 짧으면 한 목적지의 계절성과 성장률을 안정적으로 계산하기 어렵기 때문이다. 아래 표는 목적지를 보유한 월 수에 따라 나눈 결과이며, 표 아래에서 실제 모델 대상과 주의점을 설명한다.

In [3]:
bands = query('profile_sample_bands')
display(bands)
profile = pd.read_sql(text('SELECT * FROM v_demand_destination_profile'), engine)
print('유니버스(>=24개월):', int((profile['in_modeling_universe'] == 1).sum()), '목적지')


,sample_band,destinations,passengers,passenger_share_pct
0,ge36,149,152048353.0,97.9
1,24_35,20,1807371.0,1.2
2,12_23,29,937481.0,0.6
3,lt12,88,498668.0,0.3


유니버스(>=24개월): 169 목적지


### 169개 목적지를 선택한 이유

`ge36`은 36개월 이상, `24_35`는 24–35개월 기록이 있는 목적지다. 모델 대상 <strong>169곳</strong>이 전체 국제선 여객의 <strong>99.1%</strong> 를 차지한다. 다만 `24_35`는 계절 관측이 적으므로 개별 후보 해석 시 `sample_band`를 함께 확인한다.

---

## 3. 기준월까지의 정보로 만든 월간 입력

- <strong>설계</strong>: 예측 기준월을 한 달씩 앞으로 옮기면서, 그달까지 알 수 있었던 자료만으로 다음 1–5개월을 예측했다. 이를 롤링 방식이라고 한다. 시간 순서를 지켜야 하므로 데이터를 무작위로 섞지 않았다.

### 과거 입력(훈련에 해당) 및 검증 롤링 구성

| 검증 차수 | 예측 기준월 | 과거 입력 범위(B1.5는 훈련 범위) | 검증 대상월 |
|---|---|---|---|
| 1 | 2025년 1월 | 2023년 1월–2025년 1월 | 2025년 2월–6월 |
| 2 | 2025년 2월 | 2023년 1월–2025년 2월 | 2025년 3월–7월 |
| 3 | 2025년 3월 | 2023년 1월–2025년 3월 | 2025년 4월–8월 |
| 4 | 2025년 4월 | 2023년 1월–2025년 4월 | 2025년 5월–9월 |
| 5 | 2025년 5월 | 2023년 1월–2025년 5월 | 2025년 6월–10월 |
| 6 | 2025년 6월 | 2023년 1월–2025년 6월 | 2025년 7월–11월 |
| 7 | 2025년 7월 | 2023년 1월–2025년 7월 | 2025년 8월–12월 |

각 차수는 기준월까지의 자료만 사용한다. B1은 이 자료로 값을 계산하고, B1.5 이후 모델은 정답이 확인된 과거 행을 훈련에 사용한다.

- <strong>입력값</strong>: 전년동월, 최근 3개월과 전년 동기, 이전 완전연도의 계절 패턴이다.
- <strong>2025년부터 검증한 이유</strong>: 2023년은 회복 반등기라 정상적인 계절·성장 기준으로 쓰기 어려웠다.
- <strong>모델 차이</strong>: B0·B1은 계산식, B1.5 이후는 기준월마다 다시 훈련하는 모델이다.

### 실제 입력 데이터 확인

아래에서 원자료의 `shape`·`head()`·`info()`·`describe()`와 미래정보 누수 여부를 확인한다.

In [4]:
panel = fl.load_panel(engine)

# 실제 모델 입력 원자료의 구조와 분포 확인
print('원자료 shape:', panel.shape)
print('원자료 기간:', panel['period'].min(), '–', panel['period'].max(),
      '| 목적지:', panel['dest'].nunique())
display(panel[['dest', 'period', 'passengers', 'operations', 'oac']].head(10))


원자료 shape: (7825, 5)
원자료 기간: 2023-01 – 2026-07 | 목적지: 286


,dest,period,passengers,operations,oac
0,간사이,2023-01,201845.0,1046.0,6.0
1,과달라하라,2023-01,0.0,12.0,1.0
2,괌,2023-01,41882.0,226.0,2.0
3,광저우,2023-01,2343.0,43.0,1.0
4,구마모도,2023-01,1785.0,12.0,1.0
5,그린빌스파르탄버그,2023-01,0.0,5.0,1.0
6,껀터,2023-01,2200.0,13.0,1.0
7,나고야,2023-01,17353.0,127.0,1.0
8,나트랑캄란,2023-01,54258.0,298.0,3.0
9,난징,2023-01,6247.0,27.0,1.0


In [5]:
# 원자료 열 구조·자료형·결측 여부
panel.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7825 entries, 0 to 7824
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype    
---  ------      --------------  -----    
 0   dest        7825 non-null   object   
 1   period      7825 non-null   period[M]
 2   passengers  7825 non-null   float64  
 3   operations  7825 non-null   float64  
 4   oac         7825 non-null   float64  
dtypes: float64(3), object(1), period[M](1)
memory usage: 305.8+ KB


In [6]:
# 원자료 수치형 변수 기술통계
panel_describe = panel[['passengers', 'operations', 'oac']].describe().T.round(2)
display(panel_describe)


,count,mean,std,min,25%,50%,75%,max
passengers,7825.0,19845.61,40514.65,0.0,1313.0,6378.0,18783.0,383656.0
operations,7825.0,115.88,219.08,0.0,14.0,40.0,115.0,2154.0
oac,7825.0,1.46,1.06,0.0,1.0,1.0,2.0,7.0


In [7]:
# 모델 대상과 2025년 롤링 검증용 피처 생성
universe = fl.load_universe(engine)
grids = fl.build_grids(panel)
feat = fl.build_features(grids, universe, fl.VALIDATION_ORIGINS)
print('피처 shape:', feat.shape, '| 목적지', feat['dest'].nunique(),
      '| 기준월×horizon', feat.groupby(['origin','horizon']).ngroups)
print('계절지수 자기산출 비율: %.1f%%' % (feat['seasonal_index_own'].mean()*100))
print('누수 검증:', fl.leakage_checks(feat))


피처 shape: (5855, 30) | 목적지 169 | 기준월×horizon 35
계절지수 자기산출 비율: 84.9%
누수 검증: {'rows': 5855, 'max_feat<=origin': True, 'origin<target': True, 'target<=panel_end': True, 'no_future_ref': True}


In [8]:
# 검증 대상월별 기준월·horizon 커버리지
cover = (feat.groupby('tmon').agg(
    rows=('dest','size'), origins=('origin','nunique'),
    horizons=('horizon', lambda s: sorted(set(s)))))
display(cover)


,rows,origins,horizons
tmon,,,
2,166,1,[1]
3,332,2,"[1, 2]"
4,498,3,"[1, 2, 3]"
5,665,4,"[1, 2, 3, 4]"
6,833,5,"[1, 2, 3, 4, 5]"
7,836,5,"[1, 2, 3, 4, 5]"
8,839,5,"[1, 2, 3, 4, 5]"
9,673,4,"[2, 3, 4, 5]"
10,506,3,"[3, 4, 5]"


### 입력 범위와 누수 확인

원자료는 <strong>7,825행×5열</strong>, 2023년 1월–2026년 7월의 286개 목적지 자료다. 한 행은 `목적지×월`이며 여객 수·운항 횟수·출발 공항 수를 담는다. 월별 여객 수 평균이 중앙값보다 커 목적지 규모 차이가 크므로 성능은 여객량을 반영하는 WAPE로 평가한다.

24개월 이상 기록이 있는 169개 목적지만 사용한 롤링 피처는 <strong>5,855행×30열</strong>이다. 누수 검사 네 항목은 모두 `True`로, 기준월 이후 자료를 사용하지 않았다. 11·12월은 검증 반복이 적어 최종 결과에서 `저신뢰`로 표시한다.

---

## 4. 계절성과 최근 성장을 피처로 바꾸기

- <strong>계절지수</strong>: 목적지마다 어느 달에 수요가 높고 낮은지를 나타낸다. 2023년은 회복기라 계절 모양 계산에서 빼고, 2024년 이후의 완전한 연도만 사용한다. 목적지 자체 자료가 부족하면 전체 목적지의 평균 계절 모양을 대신 쓴다.
- <strong>성장계수</strong>: 최근 3개월 수요가 전년 같은 3개월보다 얼마나 늘었는지를 나타낸다. 2023년이 비교 기준에 들어가면 성장계수를 1, 즉 변화 없음으로 둔다. 그해의 회복 반등을 정상 성장으로 오해하지 않기 위해서다.

아래에서 성장률 분포, 2023년 포함 행, 극단값 제한과 계절유형 안정성을 확인한다.

In [9]:
# 성장계수 진단: 3개월 YoY의 2023 분모 영향과 안전 규칙
u = feat.drop_duplicates(['dest','origin']).copy()
display(u.groupby('origin')['growth_raw_legacy'].agg(n='count', median='median', mean='mean',
        p10=lambda s: s.quantile(.10), p90=lambda s: s.quantile(.90)).round(3))
unsafe = u['growth_denominator_has_2023']
print('2023 분모 (dest, origin): %d/%d; growth=1 중립 폴백 적용' % (unsafe.sum(), len(u)))
safe = u[u['growth_eligible']].copy()
clipped = (safe['growth_raw'] < fl.GROWTH_CLIP[0]) | (safe['growth_raw'] > fl.GROWTH_CLIP[1])
print('안전 origin의 클리핑 히트: %d/%d (%.1f%%)' % (clipped.sum(), len(safe), clipped.mean()*100))

med = safe.groupby('origin')['growth_raw'].median()
f = feat[feat['growth_eligible'] & feat['seasonal_naive_pred'].notna() & (feat['seasonal_naive_pred'] > 0)].copy()
f['med'] = f['origin'].map(med)
f['tq'] = f['tmon'].apply(lambda m: 'Q1' if m<=3 else ('Q2' if m<=6 else 'H2'))
def net_bias(df, pred): return (pred - df[A0]).sum() / df[A0].sum()
A0 = 'target_passengers'
rows = []
for w in (0.5, 0.75, 1.0):
    gw = f['med']*(1-w) + f['growth_clipped']*w
    pred = f['seasonal_naive_pred'] * gw
    rows.append({'w': w, '전체': net_bias(f, pred),
                 'Q2': net_bias(f[f.tq=='Q2'], pred[f.tq=='Q2']),
                 'H2': net_bias(f[f.tq=='H2'], pred[f.tq=='H2'])})
display(pd.DataFrame(rows).set_index('w').round(3))

# 라벨 불안정성
display(query('seasonal_label_agreement'))
# 계절형 라벨은 설명용이며 모델 입력이 아니므로 추가 임계값 튜닝은 하지 않는다.


,n,median,mean,p10,p90
origin,,,,,
2025-01,139,1.186,1.353,0.901,1.823
2025-02,139,1.090,1.221,0.875,1.649
2025-03,140,1.074,1.192,0.865,1.646
2025-04,141,1.066,1.165,0.809,1.663
2025-05,141,1.091,1.171,0.804,1.663
2025-06,142,1.069,1.142,0.758,1.607
2025-07,143,1.068,1.116,0.765,1.473


2023 분모 (dest, origin): 332/1171; growth=1 중립 폴백 적용
안전 origin의 클리핑 히트: 100/839 (11.9%)


,전체,Q2,H2
w,,,
0.50,0.022,0.022,0.022
0.75,0.020,0.021,0.020
1.00,0.018,0.021,0.017


,scope_label,comparable,same,agreement_pct
0,all,162.0,117.0,72.2
1,modeling_universe,162.0,117.0,72.2


### 회복기 성장의 과대반영 방지

성장률 비교 기준에 2023년이 포함된 조합은 <strong>332개</strong>이며, 회복 반등 때문에 과대예측을 만들 수 있다. 이 경우 성장계수를 1로 두어 전년동월 수준으로 돌아간다. 나머지 조합도 극단값을 제한한다.

계절유형 일치율은 72.2%이므로 `겨울형·여름형`은 설명에만 쓰고, 예측에는 월별 계절지수를 사용한다. 즉 2023년은 원자료에는 남기되 정상 계절·성장 기준에서는 제외한다.

---

## 5. 모델 비교

쉬운 계산식부터 학습 모델까지 같은 Rolling Validation에서 비교한다. 모델 선택에는 **전체 Validation WAPE가 가장 낮은 값**을 사용한다. 순편향과 h1–h5 오차는 선택한 모델의 특성을 이해하는 보조 지표로 함께 확인한다. Fixed Test는 선택한 모델 하나만 평가하고, 그 결과로 모델을 다시 고르거나 튜닝하지 않는다.

- **B0**: 전년동월·최근평균 기준선
- **B1**: 성장과 계절성을 반영한 계산식
- **B1.5**: 직접 Ridge 또는 B1 잔차 Ridge 보정
- **B2**: B1 잔차의 트리 보정

### 5.0 B0/B1 편향 분해 (기저 vs 성장계수)

전년동월 수요 자체가 문제인지, 여기에 곱한 성장계수가 문제인지 분리해 확인한다. `bias`가 양수면 실제보다 크게 예측했고, 음수면 작게 예측했다는 뜻이다. 아래 표에서는 전체 기간뿐 아니라 Q1·Q2·하반기의 방향이 같은지도 비교한다.

In [10]:
# 5.0 편향 분해: 전년동월 단독 / 최근평균 단독 / B1
base = feat[feat['seasonal_naive_pred'].notna() & (feat['seasonal_naive_pred'] > 0)].copy()
base['tq'] = base['tmon'].apply(lambda m: 'Q1' if m<=3 else ('Q2' if m<=6 else 'H2'))
preds = {
    'B0a_전년동월단독': base['seasonal_naive_pred'],
    'B0b_최근평균단독': base['mean_naive_pred'],
    'B1_전년동월x성장': base['seasonal_naive_pred'] * base['growth_damped'],
}
out = []
for name, p in preds.items():
    row = {'model': name}
    for seg in ('전체','Q1','Q2','H2'):
        d = base if seg=='전체' else base[base.tq==seg]
        pp = p if seg=='전체' else p[base.tq==seg]
        row[seg+'_bias'] = (pp.sum() - d['target_passengers'].sum()) / d['target_passengers'].sum()
    out.append(row)
display(pd.DataFrame(out).set_index('model').round(3))


,전체_bias,Q1_bias,Q2_bias,H2_bias
model,,,,
B0a_전년동월단독,-0.045,-0.043,-0.050,-0.043
B0b_최근평균단독,-0.013,0.112,0.036,-0.057
B1_전년동월x성장,0.002,-0.043,-0.015,0.017


#### B0를 기준선으로 두는 이유

전년동월 편향은 전 구간 −4.3%–−5.0%로 안정적이지만, 최근평균은 Q1 +11.2%로 계절 전환에 취약하다. 안전 규칙을 적용한 B1의 전체 편향은 +0.2%다. 따라서 전년동월을 기저로 유지하고, 성장률 분모에 2023년이 포함될 때만 성장계수를 1로 둔다.

### 5.1 B0 — 최근평균 / 전년동월 (하한선)

B0는 별도 학습 없이 계산하는 <strong>단순 기준선</strong>이다.

#### 기준월과 horizon

<strong>기준월(origin)</strong> 은 예측 시점, <strong>horizon(h)</strong> 은 몇 개월 뒤인지다. 기준월이 2025년 7월이면 h1은 8월, h5는 12월이다.

#### 두 B0 방식

- <strong>최근평균</strong>: 최근 3개월 평균을 h1–h5에 동일하게 적용한다. 최근 수준은 반영하지만 계절 변화는 놓칠 수 있다.
- <strong>전년동월</strong>: 예측월의 1년 전 같은 목적지 수요를 사용한다. 예: 2025년 8월은 2024년 8월 수요로 예측한다.

#### 롤링 검증 범위

7개 기준월에서 h1–h5를 예측해 `7×5=35개` 시점 조합을 평가한다. 상세 월 구성은 3절의 롤링 표와 같다.

In [11]:
# B0 예측 + WAPE. seasonal_naive=전년동월, mean_naive=최근3개월 평균
def wape_m(df, pred_col, actual='target_passengers'):
    return (df[pred_col] - df[actual]).abs().sum() / df[actual].sum()
b0 = feat.copy()
b0['pred_mean_naive'] = b0['mean_naive_pred']
b0['pred_seasonal_naive'] = b0['seasonal_naive_pred']
def summary_metrics(df, pred_col, actual='target_passengers'):
    return pd.Series({'WAPE': wape_m(df, pred_col, actual),
                      '순편향': (df[pred_col].sum()-df[actual].sum())/df[actual].sum()})
b0_summary = pd.DataFrame({
    '최근평균': summary_metrics(b0, 'pred_mean_naive'),
    '전년동월': summary_metrics(b0, 'pred_seasonal_naive'),
}).T


In [12]:
# B0 예측거리별 WAPE
b0_horizon = pd.DataFrame({
    '최근평균': b0.groupby('horizon').apply(
        lambda x: wape_m(x, 'pred_mean_naive'), include_groups=False),
    '전년동월': b0.groupby('horizon').apply(
        lambda x: wape_m(x, 'pred_seasonal_naive'), include_groups=False),
})


#### B0 오차와 예측거리

WAPE는 절대오차 합을 실제수요 합으로 나눈 통합 오차율이다. 최근평균은 15.72%, 전년동월은 14.09%다. **순편향**은 음수면 과소, 양수면 과대예측이며 두 방식 모두 전체 수요를 조금 낮게 봤다. 최근평균은 예측거리가 멀수록 나빠지지만 전년동월은 약 14%로 일정하다. 따라서 **전년동월을 B0 기준선**으로 선택한다. h1–h5 결과는 Validation 비교표에서 같은 형식으로 확인한다.

---

### 5.2 B1 — 전년동월×성장계수 · 계절지수×수준

B1도 별도 학습 없이 두 계산식을 비교한다.

#### 먼저 알아둘 성장계수 용어

- <strong>목적지 성장률</strong>: 최근 3개월 합계 ÷ 전년도 같은 3개월 합계다. 1.20은 20% 성장을 뜻한다.
- <strong>전체 목적지 중앙값</strong>: 목적지별 성장률의 가운데 값으로, 극단값에 덜 흔들리는 시장 흐름이다.
- <strong>최종 성장계수</strong>: 제한한 목적지 성장률 75%와 전체 중앙값 25%를 합친 값이다.

#### B1 후보 1. 전년동월 × 안전한 성장계수

- `목적지 성장률 = 최근 3개월 수요 합계 ÷ 전년도 같은 3개월 수요 합계`
- `최종 성장계수 = 제한한 목적지 성장률 × 0.75 + 전체 목적지 중앙값 × 0.25`
- `B1 예측 = B0 전년동월 수요 × 최종 성장계수`

예: 전년동월 10만 명, 목적지 성장률 1.20, 중앙값 1.08이면 성장계수는 1.17, 예측은 11만7천 명이다.

#### 2023년 회복기 안전장치

비교 구간에 2023년이 포함되면 성장계수를 1로 둔다. 목적지 성장률을 계산할 수 없으면 전체 중앙값을 사용한다.

#### B1 후보 2. 계절지수 × 최근 수준

최근 3개월에서 계절 효과를 제거한 기본 수준에 목표월 계절지수를 곱한다.

1. `각 최근 월의 탈계절 수준 = 해당 월 실제 수요 ÷ 해당 월 계절지수`
2. `최근 기본수준 = 최근 3개월 탈계절 수준의 평균`
3. `B1 계절지수 예측 = 최근 기본수준 × 예측 대상월 계절지수`

예: 탈계절 수준이 10만 명이고 8월 계절지수가 1.30이면 13만 명으로 예측한다.

#### 목적지 성장률·최종 성장계수·계절지수를 모두 곱하지 않는 이유

B0 전년동월에 계절성이 이미 포함되어 있어 성장계수와 계절지수를 모두 곱하면 효과가 중복될 수 있다. 따라서 두 후보를 분리해 비교한다.

In [13]:
# B1 변형 — 2023 분모 origin은 feature_layer에서 growth=1로 중립화
b1 = feat.copy()
b1['pred_b1_growth'] = b1['seasonal_naive_pred'] * b1['growth_damped']          # (i) 전년동월×성장계수
# (ii) 최근 3개월을 각 관측월 계절지수로 나눈 탈계절 수준 × 타깃월 계절지수
b1['pred_b1_index'] = b1['seasonal_level_ma_3'] * b1['seasonal_index_target']
assert b1[['pred_b1_growth', 'pred_b1_index']].notna().all().all(), 'B1 예측 결측 존재'
b1_summary = pd.DataFrame({
    '전년동월×성장': summary_metrics(b1, 'pred_b1_growth'),
    '계절지수×수준': summary_metrics(b1, 'pred_b1_index'),
}).T
b1_summary['전년동월 대비 상대개선'] = 1 - b1_summary['WAPE'] / b0_summary.loc['전년동월', 'WAPE']


In [14]:
# B1 후보 1 전체 계산 DataFrame: 전년동월 × 성장계수
b1_growth_view = b1[[
    'dest', 'origin', 'horizon', 'target_period',
    'seasonal_naive_pred', 'growth_raw', 'growth_clipped',
    'growth_damped', 'growth_denominator_has_2023', 'growth_fallback',
    'pred_b1_growth', 'target_passengers'
]].rename(columns={
    'dest': '목적지', 'origin': '기준월', 'horizon': '예측거리',
    'target_period': '예측월', 'seasonal_naive_pred': 'B0_전년동월',
    'growth_raw': '목적지_성장률', 'growth_clipped': '제한후_성장률',
    'growth_damped': '최종_성장계수',
    'growth_denominator_has_2023': '분모_2023포함',
    'growth_fallback': '성장률_대체사유',
    'pred_b1_growth': '후보1_예측', 'target_passengers': '실제수요'
})
print('후보 1 전체 계산 DataFrame shape:', b1_growth_view.shape)
display(b1_growth_view.head(10).round(3))


후보 1 전체 계산 DataFrame shape: (5855, 12)


,목적지,기준월,예측거리,예측월,B0_전년동월,목적지_성장률,제한후_성장률,최종_성장계수,분모_2023포함,성장률_대체사유,후보1_예측,실제수요
0,간사이,2025-01,1,2025-02,286775.0,NaN,1.0,1.0,True,pre_2024_denominator,286775.0,283173.0
1,간사이,2025-01,2,2025-03,314073.0,NaN,1.0,1.0,True,pre_2024_denominator,314073.0,308770.0
2,간사이,2025-01,3,2025-04,273263.0,NaN,1.0,1.0,True,pre_2024_denominator,273263.0,285882.0
3,간사이,2025-01,4,2025-05,298166.0,NaN,1.0,1.0,True,pre_2024_denominator,298166.0,303916.0
4,간사이,2025-01,5,2025-06,281147.0,NaN,1.0,1.0,True,pre_2024_denominator,281147.0,277776.0
5,과달라하라,2025-01,1,2025-02,0.0,NaN,1.0,1.0,True,pre_2024_denominator,0.0,0.0
6,과달라하라,2025-01,2,2025-03,0.0,NaN,1.0,1.0,True,pre_2024_denominator,0.0,0.0
7,과달라하라,2025-01,3,2025-04,0.0,NaN,1.0,1.0,True,pre_2024_denominator,0.0,0.0
8,과달라하라,2025-01,4,2025-05,0.0,NaN,1.0,1.0,True,pre_2024_denominator,0.0,0.0
9,과달라하라,2025-01,5,2025-06,0.0,NaN,1.0,1.0,True,pre_2024_denominator,0.0,0.0


In [15]:
# B1 후보 2 전체 계산 DataFrame: 계절지수 × 최근 수준
b1_index_view = b1[[
    'dest', 'origin', 'horizon', 'target_period',
    'seasonal_level_ma_3', 'seasonal_index_target',
    'seasonal_index_own', 'pred_b1_index', 'target_passengers'
]].rename(columns={
    'dest': '목적지', 'origin': '기준월', 'horizon': '예측거리',
    'target_period': '예측월', 'seasonal_level_ma_3': '최근_탈계절수준',
    'seasonal_index_target': '목표월_계절지수',
    'seasonal_index_own': '자체_계절지수',
    'pred_b1_index': '후보2_예측', 'target_passengers': '실제수요'
})
print('후보 2 전체 계산 DataFrame shape:', b1_index_view.shape)
display(b1_index_view.head(10).round(3))


후보 2 전체 계산 DataFrame shape: (5855, 9)


,목적지,기준월,예측거리,예측월,최근_탈계절수준,목표월_계절지수,자체_계절지수,후보2_예측,실제수요
0,간사이,2025-01,1,2025-02,298497.206,0.983,True,293504.063,283173.0
1,간사이,2025-01,2,2025-03,298497.206,1.077,True,321442.600,308770.0
2,간사이,2025-01,3,2025-04,298497.206,0.937,True,279675.010,285882.0
3,간사이,2025-01,4,2025-05,298497.206,1.022,True,305162.349,303916.0
4,간사이,2025-01,5,2025-06,298497.206,0.964,True,287744.005,277776.0
5,과달라하라,2025-01,1,2025-02,0.000,1.016,False,0.000,0.0
6,과달라하라,2025-01,2,2025-03,0.000,0.775,False,0.000,0.0
7,과달라하라,2025-01,3,2025-04,0.000,0.861,False,0.000,0.0
8,과달라하라,2025-01,4,2025-05,0.000,0.920,False,0.000,0.0
9,과달라하라,2025-01,5,2025-06,0.000,0.933,False,0.000,0.0


In [16]:
# 실제 데이터 계산 예시 1: 간사이, 기준월 2025년 7월, 후보 1
b1_example = (b1[(b1['dest'] == '간사이') &
                 (b1['origin'] == pd.Period('2025-07', freq='M'))]
              .sort_values('horizon').copy())
candidate1_example = b1_example[[
    'target_period', 'seasonal_naive_pred', 'growth_raw', 'growth_damped',
    'pred_b1_growth', 'target_passengers'
]].rename(columns={
    'target_period': '예측월', 'seasonal_naive_pred': 'B0_전년동월',
    'growth_raw': '목적지_성장률', 'growth_damped': '최종_성장계수',
    'pred_b1_growth': '후보1_예측', 'target_passengers': '실제수요'
})
candidate1_example['오차'] = candidate1_example['후보1_예측'] - candidate1_example['실제수요']
candidate1_example['절대오차'] = candidate1_example['오차'].abs()
candidate1_example['행별_APE'] = (
    candidate1_example['절대오차'] / candidate1_example['실제수요']
)
display(candidate1_example.round(3))


,예측월,B0_전년동월,목적지_성장률,최종_성장계수,후보1_예측,실제수요,오차,절대오차,행별_APE
5010,2025-08,250826.0,0.976,0.999,250604.634,269618.0,-19013.366,19013.366,0.071
5011,2025-09,273192.0,0.976,0.999,272950.895,285376.0,-12425.105,12425.105,0.044
5012,2025-10,307436.0,0.976,0.999,307164.673,315375.0,-8210.327,8210.327,0.026
5013,2025-11,319023.0,0.976,0.999,318741.447,347246.0,-28504.553,28504.553,0.082
5014,2025-12,317043.0,0.976,0.999,316763.194,362453.0,-45689.806,45689.806,0.126


In [17]:
# 간사이 후보 1 예시의 2025년 8월–12월 통합 WAPE
candidate1_example_wape = pd.DataFrame([{
    '목적지': '간사이',
    '기준월': '2025년 7월',
    '평가기간': '2025년 8월–12월',
    '절대오차합': candidate1_example['절대오차'].sum(),
    '실제수요합': candidate1_example['실제수요'].sum(),
    '통합_WAPE': (candidate1_example['절대오차'].sum() /
                 candidate1_example['실제수요'].sum())
}])
display(candidate1_example_wape.round(4))


,목적지,기준월,평가기간,절대오차합,실제수요합,통합_WAPE
0,간사이,2025년 7월,2025년 8월–12월,113843.1565,1580068.0,0.072


In [18]:
# 실제 데이터 계산 예시 2: 간사이, 기준월 2025년 7월, 후보 2
candidate2_example = b1_example[[
    'target_period', 'seasonal_level_ma_3', 'seasonal_index_target',
    'pred_b1_index', 'target_passengers'
]].rename(columns={
    'target_period': '예측월', 'seasonal_level_ma_3': '최근_탈계절수준',
    'seasonal_index_target': '목표월_계절지수',
    'pred_b1_index': '후보2_예측', 'target_passengers': '실제수요'
})
display(candidate2_example.round(3))


,예측월,최근_탈계절수준,목표월_계절지수,후보2_예측,실제수요
5010,2025-08,284390.407,0.860,244579.501,269618.0
5011,2025-09,284390.407,0.937,266388.505,285376.0
5012,2025-10,284390.407,1.054,299779.702,315375.0
5013,2025-11,284390.407,1.094,311078.143,347246.0
5014,2025-12,284390.407,1.087,309147.452,362453.0


#### B1 계산식 선택

두 후보의 WAPE는 모두 11.84%로 B0 전년동월의 14.09%보다 약 15.95% 낮다. 전년동월×성장의 순편향은 거의 0이고, 계절지수×수준은 +2.39%로 조금 높게 예측했다. 예측거리별로는 전년동월×성장은 h1–h5에서 B0보다 WAPE가 낮은 반면 계절지수×수준은 h5에서 B0보다 높다.

2025년 7월 오사카 간사이 예시의 최종 성장계수는 0.999다. 8월 B0 250,826명은 약 250,605명으로 조정됐고, 8월–12월 통합 WAPE는 7.2%다. 이는 한 목적지 예시이므로 전체 WAPE와 직접 비교하지 않는다.

최종 B1은 **전년동월×성장계수**다. 이후 학습 모델이 넘어야 할 기준으로 사용한다. 모든 모델의 전체·horizon별 WAPE는 Validation 비교표에 정리한다.

---

### 5.3 B1.5 — Ridge 선형회귀


B1.5는 여러 과거 수요 변수와 실제 수요의 관계를 Ridge가 직접 학습한다.

#### Ridge 선형회귀란

Ridge는 선형회귀 계수가 지나치게 커지지 않도록 벌점을 주는 모델이다. 비슷한 lag·이동평균 변수가 많을 때 흔들림을 줄이며 `alpha=1.0`을 사용한다.

#### B1.5가 사용하는 입력 변수

- <strong>lag_1·lag_3·lag_12</strong>: 기준월, 2개월 전, 11개월 전 수요다.
- <strong>ma_3·ma_12</strong>: 기준월을 포함한 최근 3개월·12개월 평균 수요다.
- <strong>seasonal_index_target</strong>: 예측 대상월의 계절지수다.
- <strong>growth_damped</strong>: B1에서 사용한 안전한 최종 성장계수다.
- <strong>horizon</strong>: 기준월에서 몇 개월 뒤를 예측하는지다.
- <strong>목표 월 표시값</strong>: 월별 차이를 학습하는 0·1 변수다.

수요 변수는 `log(1+x)`로 변환하고 훈련자료 기준으로 표준화한다.

#### 실제 훈련과 롤링 검증 과정

1. 기준월까지 실제 수요가 확인된 과거 행만 훈련한다.
2. 같은 기준월의 h1–h5를 검증한다.
3. 기준월이 이동하면 새 정답을 추가해 Ridge와 스케일러를 다시 적합한다.

즉 7개 기준월마다 다시 학습하는 <strong>walk-forward 모델</strong>이며 미래 실제값은 사용하지 않는다.


In [19]:
# B1.5 선형회귀 (walk-forward Ridge). 폴드별 학습 구간(target_period <= origin)에서만 적합.
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

allf = fl.build_features(grids, universe, pd.period_range('2024-06', '2025-07', freq='M'))  # 학습 풀
B15_NUM = ['lag_1','lag_3','lag_12','ma_3','ma_12','seasonal_index_target','growth_damped','horizon']
B15_LOG = ['lag_1','lag_3','lag_12','ma_3','ma_12']

def _b15_Xy(df):
    X = df[B15_NUM].copy()
    for c in B15_LOG:
        X[c] = np.log1p(X[c].clip(lower=0))
    md = pd.get_dummies(df['target_month'].astype(int), prefix='m').reindex(
        columns=[f'm_{m}' for m in range(2, 13)], fill_value=0)   # m_1 기준월
    X = pd.concat([X.reset_index(drop=True), md.reset_index(drop=True)], axis=1)
    return X, np.log1p(df['target_passengers'].to_numpy())

_parts = []
_fold_rows = []
for o in fl.VALIDATION_ORIGINS:
    tr, te = allf[allf['target_period'] <= o], allf[allf['origin'] == o]   # 누수 통제: 관측된 타깃만 학습
    Xtr, ytr = _b15_Xy(tr); Xte, _ = _b15_Xy(te)
    Xte = Xte.reindex(columns=Xtr.columns, fill_value=0)
    sc = StandardScaler().fit(Xtr.values)                                  # 스케일러도 학습 구간만
    mdl = Ridge(alpha=1.0).fit(sc.transform(Xtr.values), ytr)             # 결정적 -> 시드 불필요
    t = te.copy(); t['pred_b15'] = np.clip(
        np.expm1(mdl.predict(sc.transform(Xte.values))), 0, None)
    _parts.append(t)
    _fold_rows.append({
        '기준월': o, '훈련_행수': len(tr),
        '훈련_예측월_시작': tr['target_period'].min(),
        '훈련_예측월_종료': tr['target_period'].max(),
        '검증_행수': len(te),
        '검증_예측월_시작': te['target_period'].min(),
        '검증_예측월_종료': te['target_period'].max(),
    })
b15 = pd.concat(_parts, ignore_index=True)
b15_folds = pd.DataFrame(_fold_rows)
b15['오차'] = b15['pred_b15'] - b15['target_passengers']
b15['절대오차'] = b15['오차'].abs()
b15['행별_APE'] = np.where(
    b15['target_passengers'] > 0,
    b15['절대오차'] / b15['target_passengers'], np.nan)
b15['tq'] = b15['tmon'].apply(
    lambda m: 'Q1' if m <= 3 else ('Q2' if m <= 6 else 'H2'))

def _W(df, col, actual='target_passengers'):
    return (df[col] - df[actual]).abs().sum() / df[actual].sum()

def _bias_b15(df):
    return ((df['pred_b15'].sum() - df['target_passengers'].sum()) /
            df['target_passengers'].sum())


In [20]:
# B1.5 입력 변수 설명
b15_feature_guide = pd.DataFrame([
    ['lag_1', '기준월 수요', 'log(1+x) 후 표준화'],
    ['lag_3', '기준월에서 두 달 전 수요', 'log(1+x) 후 표준화'],
    ['lag_12', '기준월에서 11개월 전 수요', 'log(1+x) 후 표준화'],
    ['ma_3', '기준월 포함 최근 3개월 평균', 'log(1+x) 후 표준화'],
    ['ma_12', '기준월 포함 최근 12개월 평균', 'log(1+x) 후 표준화'],
    ['seasonal_index_target', '예측 대상월 계절지수', '표준화'],
    ['growth_damped', '안전하게 조정한 최종 성장계수', '표준화'],
    ['horizon', '몇 개월 뒤 예측인지', '표준화'],
    ['m_2–m_12', '예측 대상월 표시값, 1월은 기준', '0 또는 1'],
], columns=['입력변수', '의미', '모델 입력 처리'])
display(b15_feature_guide)


,입력변수,의미,모델 입력 처리
0,lag_1,기준월 수요,log(1+x) 후 표준화
1,lag_3,기준월에서 두 달 전 수요,log(1+x) 후 표준화
2,lag_12,기준월에서 11개월 전 수요,log(1+x) 후 표준화
3,ma_3,기준월 포함 최근 3개월 평균,log(1+x) 후 표준화
4,ma_12,기준월 포함 최근 12개월 평균,log(1+x) 후 표준화
5,seasonal_index_target,예측 대상월 계절지수,표준화
6,growth_damped,안전하게 조정한 최종 성장계수,표준화
7,horizon,몇 개월 뒤 예측인지,표준화
8,m_2–m_12,"예측 대상월 표시값, 1월은 기준",0 또는 1


In [21]:
# 7개 차수별 실제 훈련·검증 범위
display(b15_folds)


,기준월,훈련_행수,훈련_예측월_시작,훈련_예측월_종료,검증_행수,검증_예측월_시작,검증_예측월_종료
0,2025-01,4052,2024-07,2025-01,830,2025-02,2025-06
1,2025-02,4878,2024-07,2025-02,830,2025-03,2025-07
2,2025-03,5706,2024-07,2025-03,830,2025-04,2025-08
3,2025-04,6535,2024-07,2025-04,835,2025-05,2025-09
4,2025-05,7366,2024-07,2025-05,840,2025-06,2025-10
5,2025-06,8199,2024-07,2025-06,845,2025-07,2025-11
6,2025-07,9035,2024-07,2025-07,845,2025-08,2025-12


In [22]:
# B1.5 훈련 후보 풀 전체 DataFrame
b15_training_pool = allf[[
    'dest', 'origin', 'target_period',
    *B15_NUM, 'target_month', 'target_passengers'
]].rename(columns={
    'dest': '목적지', 'origin': '피처_기준월',
    'horizon': '예측거리', 'target_period': '예측월',
    'target_month': '예측월_숫자', 'target_passengers': '실제수요'
})
print('B1.5 훈련 후보 풀 shape:', b15_training_pool.shape)
display(b15_training_pool.round(3))


B1.5 훈련 후보 풀 shape: (11560, 13)


,목적지,피처_기준월,예측월,lag_1,lag_3,lag_12,ma_3,ma_12,seasonal_index_target,growth_damped,예측거리,예측월_숫자,실제수요
0,간사이,2024-06,2024-07,281147.0,273263.0,258925.0,284192.0,284960.667,1.000,1.000,1,7,277652.0
1,간사이,2024-06,2024-08,281147.0,273263.0,258925.0,284192.0,284960.667,1.000,1.000,2,8,250826.0
2,간사이,2024-06,2024-09,281147.0,273263.0,258925.0,284192.0,284960.667,1.000,1.000,3,9,273192.0
3,간사이,2024-06,2024-10,281147.0,273263.0,258925.0,284192.0,284960.667,1.000,1.000,4,10,307436.0
4,간사이,2024-06,2024-11,281147.0,273263.0,258925.0,284192.0,284960.667,1.000,1.000,5,11,319023.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
11555,어저우화후,2025-07,2025-08,0.0,0.0,0.0,0.0,0.000,1.081,1.068,1,8,0.0
11556,어저우화후,2025-07,2025-09,0.0,0.0,0.0,0.0,0.000,0.979,1.068,2,9,0.0
11557,어저우화후,2025-07,2025-10,0.0,0.0,0.0,0.0,0.000,1.076,1.068,3,10,0.0
11558,어저우화후,2025-07,2025-11,0.0,0.0,0.0,0.0,0.000,0.944,1.068,4,11,0.0


In [23]:
# B1.5 롤링 검증 예측 전체 DataFrame
b15_prediction_view = b15[[
    'dest', 'origin', 'horizon', 'target_period',
    'pred_b15', 'target_passengers', '오차', '절대오차', '행별_APE'
]].rename(columns={
    'dest': '목적지', 'origin': '기준월', 'horizon': '예측거리',
    'target_period': '예측월', 'pred_b15': 'B1.5_예측',
    'target_passengers': '실제수요'
})
print('B1.5 전체 예측 DataFrame shape:', b15_prediction_view.shape)
display(b15_prediction_view.round(3))


B1.5 전체 예측 DataFrame shape: (5855, 9)


,목적지,기준월,예측거리,예측월,B1.5_예측,실제수요,오차,절대오차,행별_APE
0,간사이,2025-01,1,2025-02,313917.715,283173.0,30744.715,30744.715,0.109
1,간사이,2025-01,2,2025-03,315718.867,308770.0,6948.867,6948.867,0.023
2,간사이,2025-01,3,2025-04,317530.353,285882.0,31648.353,31648.353,0.111
3,간사이,2025-01,4,2025-05,319352.233,303916.0,15436.233,15436.233,0.051
4,간사이,2025-01,5,2025-06,321184.566,277776.0,43408.566,43408.566,0.156
...,...,...,...,...,...,...,...,...,...
5850,어저우화후,2025-07,1,2025-08,0.404,0.0,0.404,0.404,NaN
5851,어저우화후,2025-07,2,2025-09,0.129,0.0,0.129,0.129,NaN
5852,어저우화후,2025-07,3,2025-10,0.280,0.0,0.280,0.280,NaN
5853,어저우화후,2025-07,4,2025-11,0.000,0.0,0.000,0.000,NaN


In [24]:
# 실제 데이터 예시: 간사이, 기준월 2025년 7월, B1.5
b15_example = b15_prediction_view[
    (b15_prediction_view['목적지'] == '간사이') &
    (b15_prediction_view['기준월'] == pd.Period('2025-07', freq='M'))
].sort_values('예측거리').copy()
display(b15_example.round(3))


,목적지,기준월,예측거리,예측월,B1.5_예측,실제수요,오차,절대오차,행별_APE
5010,간사이,2025-07,1,2025-08,280451.621,269618.0,10833.621,10833.621,0.040
5011,간사이,2025-07,2,2025-09,255398.047,285376.0,-29977.953,29977.953,0.105
5012,간사이,2025-07,3,2025-10,293511.550,315375.0,-21863.450,21863.450,0.069
5013,간사이,2025-07,4,2025-11,248266.551,347246.0,-98979.449,98979.449,0.285
5014,간사이,2025-07,5,2025-12,271992.996,362453.0,-90460.004,90460.004,0.250


In [25]:
# 간사이 B1.5 예시의 2025년 8월–12월 통합 WAPE
b15_example_wape = pd.DataFrame([{
    '목적지': '간사이', '기준월': '2025년 7월',
    '평가기간': '2025년 8월–12월',
    '절대오차합': b15_example['절대오차'].sum(),
    '실제수요합': b15_example['실제수요'].sum(),
    '통합_WAPE': b15_example['절대오차'].sum() / b15_example['실제수요'].sum(),
}])
display(b15_example_wape.round(4))


,목적지,기준월,평가기간,절대오차합,실제수요합,통합_WAPE
0,간사이,2025년 7월,2025년 8월–12월,252114.4777,1580068.0,0.1596


In [26]:
# B1.5 전체 성능과 B1 대비 승급 여부
b15_wape = _W(b15, 'pred_b15')
b1_selected_wape = b1_summary.loc['전년동월×성장', 'WAPE']
b15_rel = 1 - b15_wape / b1_selected_wape
b15_overall = pd.DataFrame([{
    'B1.5_WAPE': b15_wape,
    'B1_WAPE': b1_selected_wape,
    'B1대비_상대개선': b15_rel,
}])


In [27]:
# B1.5 기간별 순편향
b15_bias = pd.DataFrame([{
    '전체': _bias_b15(b15),
    'Q1': _bias_b15(b15[b15['tq'] == 'Q1']),
    'Q2': _bias_b15(b15[b15['tq'] == 'Q2']),
    'H2': _bias_b15(b15[b15['tq'] == 'H2']),
}])


#### 직접 학습 방식의 한계

입력 변수표는 Ridge가 한 번의 예측에 사용하는 값을 보여준다. 목적지 규모는 이름 대신 lag·이동평균으로 전달한다.

#### 실제 오사카 간사이 예시

2025년 7월 오사카 간사이 예시의 WAPE는 15.96%로 B1 7.2%보다 나빴다.

#### 결과 해석과 결론

전체 WAPE는 17.64%로 B1보다 약 49% 나빴다. B1.5는 기각하고 다음 절에서 B1의 오차만 보정한다.

---

### 5.4 B1.5-v2 — B1 잔차 Ridge 보정 모델


B1+Ridge는 수요 전체를 다시 예측하지 않고 <strong>B1이 남긴 오차만 Ridge로 보정</strong>한다. 설계와 파라미터 선택에는 2025년 Validation만 사용한다.

#### 잔차 보정 방식

1. 모든 horizon에서 `전년동월×성장계수`인 B1을 기준값으로 둔다.
2. Ridge는 `log(실제수요+1)−log(B1예측+1)`인 <strong>로그 잔차</strong>를 학습한다.
3. h1–h5별로 따로 학습하고 학습창·`alpha`·보정비율을 Validation에서 고른다.

#### 한 번의 예측이 만들어지는 계산 순서

1. B1이 기준예측을 만든다.
2. Ridge가 과거 B1 오차에서 보정값을 계산한다.
3. 보정값의 일부만 B1에 반영한다.

`최종예측=exp(log(B1예측+1)+보정비율×Ridge보정값)−1`이다. B1이 10만 명이고 Ridge가 +10%를 제안해도 보정비율 30%면 약 +3%만 반영한다.

#### 롤링 학습은 어떻게 진행되나

2025년 7월 기준 최근 3개월 창은 정답이 확인된 5–7월 오차를 학습해 8–12월을 보정한다. 원자료 자체는 전년동월·lag·성장계수 계산을 위해 2023년부터 사용한다. 기준월이 이동할 때마다 horizon별 Ridge를 다시 학습한다.


In [28]:
# B1.5-v2 공통 준비: B1을 기준점으로 두고 로그 잔차만 학습한다.
b15v2_dev = allf.copy().reset_index(drop=True)
b15v2_dev['b1_base'] = (
    b15v2_dev['seasonal_naive_pred'] * b15v2_dev['growth_damped'])
b15v2_dev['target_resid_log'] = (
    np.log1p(b15v2_dev['target_passengers'])
    - np.log1p(b15v2_dev['b1_base'].clip(lower=0)))

B15V2_NUM = [
    'log_b1_base', 'log_seasonal_naive', 'log_lag_1', 'log_lag_3',
    'log_lag_12', 'log_ma_3', 'log_ma_12', 'recent_vs_b1',
    'growth_damped', 'seasonal_index_target',
]
b15v2_dev['log_b1_base'] = np.log1p(b15v2_dev['b1_base'].clip(lower=0))
b15v2_dev['log_seasonal_naive'] = np.log1p(
    b15v2_dev['seasonal_naive_pred'].clip(lower=0))
for c in ['lag_1', 'lag_3', 'lag_12', 'ma_3', 'ma_12']:
    b15v2_dev[f'log_{c}'] = np.log1p(b15v2_dev[c].clip(lower=0))
b15v2_dev['recent_vs_b1'] = (
    b15v2_dev['log_ma_3'] - b15v2_dev['log_b1_base'])

def _b15v2_design(df, feature_mode='core'):
    xnum = df[B15V2_NUM].astype(float).copy()
    cats = pd.DataFrame({'target_month': df['target_month'].astype(str)}, index=df.index)
    if feature_mode in {'dest', 'dest_month'}:
        cats['dest'] = df['dest'].astype(str)
    if feature_mode == 'dest_month':
        cats['dest_month'] = (
            df['dest'].astype(str) + '__' + df['target_month'].astype(str))
    return xnum, pd.get_dummies(cats, drop_first=True, dtype=float)

def _b15v2_full_correction(train_window, alpha, feature_mode='core'):
    xnum_all, xd_all = _b15v2_design(b15v2_dev, feature_mode)
    parts = []
    for o in fl.VALIDATION_ORIGINS:
        trmask = b15v2_dev['target_period'] <= o
        if train_window is not None:
            trmask &= b15v2_dev['target_period'] >= (o - (train_window - 1))
        for h in fl.HORIZONS:
            htr = trmask & (b15v2_dev['horizon'] == h)
            hte = ((b15v2_dev['origin'] == o) &
                   (b15v2_dev['horizon'] == h))
            sc = StandardScaler().fit(xnum_all.loc[htr])
            xtr = np.column_stack([
                sc.transform(xnum_all.loc[htr]), xd_all.loc[htr].to_numpy()])
            xte = np.column_stack([
                sc.transform(xnum_all.loc[hte]), xd_all.loc[hte].to_numpy()])
            mdl = Ridge(alpha=alpha).fit(
                xtr, b15v2_dev.loc[htr, 'target_resid_log'])
            parts.append(pd.DataFrame({
                'row_id': b15v2_dev.index[hte],
                'correction_full': mdl.predict(xte),
            }))
    p = pd.concat(parts).set_index('row_id').sort_index()
    out = b15v2_dev.loc[p.index].copy()
    out['correction_full'] = p['correction_full']
    return out

def _b15v2_apply(out, blend):
    d = out.copy()
    d['pred_b15v2'] = np.clip(np.expm1(
        d['log_b1_base'] + blend * d['correction_full']), 0, None)
    return d

def _b15v2_metrics(d):
    w = _W(d, 'pred_b15v2')
    clean = d[d['origin'] >= pd.Period('2025-03', freq='M')]
    return w, _W(clean, 'pred_b15v2'), _bias_b15(
        d.rename(columns={'pred_b15v2': 'pred_b15'}))


In [29]:
# 1단계 튜닝: 2025 Validation에서 학습창·alpha·보정비율 비교
b15v2_tuning_rows = []
for window in [None, 3, 6, 9, 12]:
    for alpha in [10.0, 30.0, 100.0, 300.0]:
        full = _b15v2_full_correction(window, alpha, feature_mode='core')
        for blend in [0.10, 0.20, 0.30, 0.40]:
            d = _b15v2_apply(full, blend)
            w, clean_w, bias = _b15v2_metrics(d)
            b15v2_tuning_rows.append({
                '학습창': '누적' if window is None else f'최근 {window}개월',
                'window': window, 'alpha': alpha, '보정비율': blend,
                '전체_WAPE': w, '2025년3월_이후_WAPE': clean_w,
                '순편향': bias,
            })
b15v2_tuning = pd.DataFrame(b15v2_tuning_rows).sort_values('전체_WAPE')
display(b15v2_tuning.head(10).drop(columns='window').round(4))


,학습창,alpha,보정비율,전체_WAPE,2025년3월_이후_WAPE,순편향
30,최근 3개월,300.0,0.3,0.1147,0.1068,0.0059
49,최근 9개월,10.0,0.2,0.1148,0.1077,0.0057
29,최근 3개월,300.0,0.2,0.1149,0.1068,0.0038
54,최근 9개월,30.0,0.3,0.1150,0.1088,0.0081
53,최근 9개월,30.0,0.2,0.1150,0.1078,0.0051
50,최근 9개월,10.0,0.3,0.1151,0.1090,0.0089
33,최근 6개월,10.0,0.2,0.1153,0.1083,0.0040
17,최근 3개월,10.0,0.2,0.1153,0.1085,-0.0000
57,최근 9개월,100.0,0.2,0.1154,0.1082,0.0060
62,최근 9개월,300.0,0.3,0.1154,0.1087,0.0128


#### Ridge 설정 선택

학습창 5개×`alpha` 4개×보정비율 4개, 총 80개 조합을 Validation에서 비교했다. `alpha`는 보정의 안정성, 보정비율은 Ridge 보정을 실제 예측에 반영하는 비율이다.

최선은 <strong>최근 3개월·alpha 300·보정비율 30%</strong> 로, 전체 WAPE는 11.47%, 2025년 3월 이후는 10.68%였다. 최근 오차만 작게 보정하는 방식이 가장 안정적이었으며 이 결과는 Test가 아닌 파라미터 선택용 Validation 점수다.

In [30]:
# 2단계 튜닝: 선택된 파라미터에서 목적지 정보 추가 효과 비교
b15v2_structure_rows, b15v2_structure_predictions = [], {}
for mode, label in [('core', '공통'), ('dest', '목적지'),
                    ('dest_month', '목적지×월')]:
    full = _b15v2_full_correction(3, 300.0, feature_mode=mode)
    d = _b15v2_apply(full, 0.30)
    w, clean_w, bias = _b15v2_metrics(d)
    b15v2_structure_rows.append({
        '입력구조': label, '전체_WAPE': w,
        '2025년3월_이후_WAPE': clean_w, '순편향': bias})
    b15v2_structure_predictions[mode] = d
b15v2_structure = pd.DataFrame(b15v2_structure_rows).sort_values('전체_WAPE')
display(b15v2_structure.round(4))


,입력구조,전체_WAPE,2025년3월_이후_WAPE,순편향
2,목적지×월,0.1147,0.1068,0.0060
1,목적지,0.1147,0.1068,0.0059
0,공통,0.1147,0.1068,0.0059


#### 목적지 정보 비교 결과

공통·목적지·목적지×월 구조의 WAPE는 모두 11.47% 수준이었다. 목적지×월의 추가 개선이 약 0.004%p에 불과해 가장 단순한 `공통` 구조를 선택한다. 예측은 목적지별로 만들되 Ridge 계수만 함께 공유하는 방식이다.

In [31]:
# 최종 Validation 후보: 목적지 추가 이득이 0.01%p 미만이라 단순한 공통 구조를 유지
b15v2 = b15v2_structure_predictions['core'].copy()
b15v2['오차'] = b15v2['pred_b15v2'] - b15v2['target_passengers']
b15v2['절대오차'] = b15v2['오차'].abs()
b15v2_wape = _W(b15v2, 'pred_b15v2')
b15v2_rel = 1 - b15v2_wape / b1_selected_wape
b15v2_overall = pd.DataFrame([{
    'B1_WAPE': b1_selected_wape, 'B1.5-v2_WAPE': b15v2_wape,
    'B1대비_상대개선': b15v2_rel,
    '순편향': (b15v2['pred_b15v2'].sum() -
             b15v2['target_passengers'].sum()) / b15v2['target_passengers'].sum(),
}])


#### Ridge 잔차 보정 성능

Validation WAPE는 B1 11.84%에서 Ridge 잔차보정 11.47%로 0.37%p 낮아졌다. 상대개선은 3.11%, 순편향은 +0.59%다. 모델별 전체·예측거리별 오차는 마지막 비교표에서 함께 확인한다.

In [32]:
# B1.5-v2의 horizon별·기준월별 Validation 안정성
b15v2_by_h = b15v2.groupby('horizon').apply(
    lambda x: _W(x, 'pred_b15v2'), include_groups=False).rename('B1.5-v2_WAPE')
b15v2_by_origin = b15v2.groupby('origin').apply(lambda x: pd.Series({
    'B1_WAPE': _W(x, 'b1_base'),
    'B1.5-v2_WAPE': _W(x, 'pred_b15v2'),
    '상대개선': 1 - _W(x, 'pred_b15v2') / _W(x, 'b1_base'),
}), include_groups=False)
print('B1보다 개선된 기준월:',
      f"{int((b15v2_by_origin['상대개선'] > 0).sum())}/{len(b15v2_by_origin)}")


B1보다 개선된 기준월: 6/7


#### 다음 비교

Ridge 잔차보정은 직접 Ridge보다 Validation WAPE가 낮다. 이어서 같은 B1 기준값을 트리 모델로 보정했을 때 추가 개선이 있는지 확인한다.

### 5.5 B2 — B1 잔차 트리 보정 모델 비교


B1.5-v2와 마찬가지로 B1 예측을 출발점으로 두고 남은 로그 오차만 보정한다. Ridge 대신 <strong>HistGradientBoosting·LightGBM·XGBoost</strong>를 같은 조건에서 비교해 비선형 관계가 추가 개선을 만드는지 확인한다.

세 모델은 같은 공통 입력과 h1–h5별 학습 구조를 사용한다. 학습창은 최근 3·6·12개월과 누적, 보정비율은 10–40%만 비교하며 트리 구조는 결과를 보기 전에 모델별 한 세트로 고정한다. 모델 선택은 전체 Validation WAPE를 기준으로 하며, 순편향과 horizon별 오차도 함께 확인한다.


In [33]:
# B2: B1 로그 잔차를 세 트리 모델로 보정하고 동일한 롤링 Validation에서 비교한다.
from sklearn.ensemble import HistGradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

B2_WINDOWS = [('최근 3개월', 3), ('최근 6개월', 6),
              ('최근 12개월', 12), ('누적', None)]
B2_BLENDS = [0.10, 0.20, 0.30, 0.40]
B2_MODEL_LABELS = {
    'histgb': 'HistGradientBoosting',
    'lgbm': 'LightGBM',
    'xgb': 'XGBoost',
}

def _make_b2_model(name):
    if name == 'histgb':
        return HistGradientBoostingRegressor(
            learning_rate=0.04, max_iter=250, max_leaf_nodes=15,
            min_samples_leaf=40, l2_regularization=10.0, random_state=42)
    if name == 'lgbm':
        return LGBMRegressor(
            objective='regression_l2', learning_rate=0.03, n_estimators=250,
            num_leaves=15, min_child_samples=40, subsample=0.90,
            subsample_freq=1, colsample_bytree=0.90, reg_alpha=1.0,
            reg_lambda=10.0, verbosity=-1, n_jobs=-1, random_state=42)
    if name == 'xgb':
        return XGBRegressor(
            objective='reg:squarederror', learning_rate=0.03,
            n_estimators=250, max_depth=3, min_child_weight=20,
            subsample=0.90, colsample_bytree=0.90, reg_alpha=1.0,
            reg_lambda=10.0, tree_method='hist', n_jobs=-1, random_state=42)
    raise ValueError(f'알 수 없는 B2 모델: {name}')

b2_xnum, b2_xdummy = _b15v2_design(b15v2_dev, feature_mode='core')
b2_x = pd.concat([b2_xnum, b2_xdummy], axis=1).astype(float)

def _b2_full_correction(model_name, train_window):
    parts = []
    for o in fl.VALIDATION_ORIGINS:
        trmask = b15v2_dev['target_period'] <= o
        if train_window is not None:
            trmask &= b15v2_dev['target_period'] >= (o - (train_window - 1))
        for h in fl.HORIZONS:
            htr = trmask & (b15v2_dev['horizon'] == h)
            hte = ((b15v2_dev['origin'] == o) &
                   (b15v2_dev['horizon'] == h))
            model = _make_b2_model(model_name)
            model.fit(b2_x.loc[htr], b15v2_dev.loc[htr, 'target_resid_log'])
            parts.append(pd.DataFrame({
                'row_id': b15v2_dev.index[hte],
                'correction_full': model.predict(b2_x.loc[hte]),
            }))
    pred = pd.concat(parts).set_index('row_id').sort_index()
    out = b15v2_dev.loc[pred.index].copy()
    out['correction_full'] = pred['correction_full']
    return out

def _b2_apply(full, blend):
    out = full.copy()
    out['pred_b2'] = np.clip(np.expm1(
        out['log_b1_base'] + blend * out['correction_full']), 0, None)
    return out

# 모델별로 학습창을 한 번 적합하고, 보정비율은 같은 예측에서 비교한다.
b2_tuning_rows, b2_full_cache = [], {}
for model_name, model_label in B2_MODEL_LABELS.items():
    for window_label, window in B2_WINDOWS:
        full = _b2_full_correction(model_name, window)
        b2_full_cache[(model_name, window_label)] = full
        for blend in B2_BLENDS:
            out = _b2_apply(full, blend)
            clean = out[out['origin'] >= pd.Period('2025-03', freq='M')]
            b2_tuning_rows.append({
                'model_key': model_name, '모델': model_label,
                '학습창': window_label, '보정비율': blend,
                '전체_WAPE': _W(out, 'pred_b2'),
                '2025년3월_이후_WAPE': _W(clean, 'pred_b2'),
                '순편향': ((out['pred_b2'].sum() - out['target_passengers'].sum())
                         / out['target_passengers'].sum()),
            })

b2_tuning = pd.DataFrame(b2_tuning_rows).sort_values('전체_WAPE')
b2_best = (b2_tuning.sort_values('전체_WAPE')
           .groupby('model_key', as_index=False).first()
           .sort_values('전체_WAPE'))
display(b2_best.drop(columns='model_key').round(4))

# 최선 설정의 예측을 복원하고 B1·Ridge 보정과 한 표에서 비교한다.
b2_predictions = {}
for row in b2_best.itertuples(index=False):
    b2_predictions[row.model_key] = _b2_apply(
        b2_full_cache[(row.model_key, row.학습창)], row.보정비율)

comparison_rows = [
    {'모델': 'B1 전년동월×성장', 'WAPE': b1_selected_wape,
     '순편향': ((b15v2['b1_base'].sum() - b15v2['target_passengers'].sum())
              / b15v2['target_passengers'].sum())},
    {'모델': 'B1+Ridge 잔차보정', 'WAPE': b15v2_wape,
     '순편향': b15v2_overall.loc[0, '순편향']},
]
for model_key, out in b2_predictions.items():
    comparison_rows.append({
        '모델': f"B1+{B2_MODEL_LABELS[model_key]}",
        'WAPE': _W(out, 'pred_b2'),
        '순편향': ((out['pred_b2'].sum() - out['target_passengers'].sum())
                 / out['target_passengers'].sum()),
    })
b2_comparison = pd.DataFrame(comparison_rows)
b2_comparison['B1대비_상대개선'] = 1 - b2_comparison['WAPE'] / b1_selected_wape


,모델,학습창,보정비율,전체_WAPE,2025년3월_이후_WAPE,순편향
2,XGBoost,최근 3개월,0.2,0.1157,0.1078,0.0015
1,LightGBM,최근 12개월,0.2,0.1159,0.1087,0.0077
0,HistGradientBoosting,누적,0.2,0.1159,0.1087,0.0085


### 5.6 Validation 성능 비교와 모델 선택

모든 모델을 같은 2025년 Rolling Validation 행에서 평가한다. 전체 WAPE를 **낮은 순서**로 정렬하고, 순편향과 예측거리 h1–h5의 WAPE를 같은 열에서 비교한다. 첫 행의 모델을 Fixed Test 대상으로 확정한다.

In [34]:
# 같은 평가 행·분모에서 계산한 Validation WAPE를 오름차순으로 정렬한다.
validation_specs = [
    ('B0 최근평균', b0, 'pred_mean_naive'),
    ('B0 전년동월', b0, 'pred_seasonal_naive'),
    ('B1 전년동월×성장', b1, 'pred_b1_growth'),
    ('B1 계절지수×수준', b1, 'pred_b1_index'),
    ('B1.5 직접 Ridge', b15, 'pred_b15'),
    ('B1+Ridge 잔차보정', b15v2, 'pred_b15v2'),
]
validation_specs += [
    (f'B1+{B2_MODEL_LABELS[key]}', frame, 'pred_b2')
    for key, frame in b2_predictions.items()
]
validation_rows = []
key_cols = ['dest', 'origin', 'horizon']
reference_keys = set(map(tuple, feat[key_cols].itertuples(index=False, name=None)))
for model_name, frame, pred_col in validation_specs:
    assert len(frame) == len(feat)
    assert not frame.duplicated(key_cols).any()
    assert set(map(tuple, frame[key_cols].itertuples(index=False, name=None))) == reference_keys
    metrics = summary_metrics(frame, pred_col)
    row = {'모델': model_name, 'Validation WAPE': metrics['WAPE'],
           '순편향': metrics['순편향']}
    row.update({f'h{h}': _W(frame[frame['horizon'] == h], pred_col)
                for h in fl.HORIZONS})
    validation_rows.append(row)
validation_summary = (pd.DataFrame(validation_rows)
                      .sort_values('Validation WAPE', kind='stable')
                      .reset_index(drop=True))
FINAL_VALIDATION_MODEL = validation_summary.loc[0, '모델']
FINAL_VALIDATION_WAPE = validation_summary.loc[0, 'Validation WAPE']
view = validation_summary.copy()
view.insert(0, '선택', np.where(view.index == 0, '✓', ''))
for col in ['Validation WAPE', '순편향', *[f'h{h}' for h in fl.HORIZONS]]:
    view[col] = view[col].map(lambda x: f'{x:.2%}')
display(view)
print('선택 모델:', FINAL_VALIDATION_MODEL,
      '| Validation WAPE:', f'{FINAL_VALIDATION_WAPE:.2%}')
assert FINAL_VALIDATION_MODEL == 'B1+Ridge 잔차보정'
assert np.isclose(FINAL_VALIDATION_WAPE, 0.11472950, atol=1e-7)

,선택,모델,Validation WAPE,순편향,h1,h2,h3,h4,h5
0,✓,B1+Ridge 잔차보정,11.47%,0.59%,9.80%,11.05%,11.80%,12.28%,12.36%
1,,B1+XGBoost,11.57%,0.15%,9.85%,11.11%,11.83%,12.24%,12.75%
2,,B1+LightGBM,11.59%,0.77%,9.65%,10.93%,11.88%,12.51%,12.89%
3,,B1+HistGradientBoosting,11.59%,0.85%,9.59%,10.94%,11.85%,12.50%,13.00%
4,,B1 계절지수×수준,11.84%,2.39%,9.18%,10.78%,11.79%,13.00%,14.32%
5,,B1 전년동월×성장,11.84%,0.02%,10.31%,11.37%,11.99%,12.48%,12.99%
6,,B0 전년동월,14.09%,-4.66%,13.95%,14.17%,14.10%,14.08%,14.15%
7,,B0 최근평균,15.72%,-1.29%,12.79%,15.18%,16.06%,16.85%,17.59%
8,,B1.5 직접 Ridge,17.64%,8.34%,14.39%,17.01%,18.74%,18.64%,19.30%


선택 모델: B1+Ridge 잔차보정 | Validation WAPE: 11.47%


Ridge 잔차보정의 Validation WAPE는 **11.47%** 로 비교 모델 중 가장 낮다. B1의 11.84%보다 0.37%p 낮으며, 이 기준에 따라 **B1+Ridge 잔차보정**을 Fixed Test 대상으로 선택한다. Test 값은 이 선택에 사용하지 않는다.

## 6. 2026년 Fixed Test — 선택 모델만 평가

Validation에서 선택한 Ridge 잔차보정의 학습창·벌점·보정비율을 고정한다. 기준월은 2025년 8월–2026년 6월, 대상월은 2026년 1–7월이다. 예측거리 h1–h5를 유지하고 Test에서 설정을 다시 고르지 않는다.

In [35]:
# Validation WAPE 최저 모델 하나만 Fixed Test에 적용한다.
assert FINAL_VALIDATION_MODEL == 'B1+Ridge 잔차보정' 
FINAL_TEST_MODEL = FINAL_VALIDATION_MODEL
FINAL_TEST_ORIGINS = pd.period_range('2025-08', '2026-06', freq='M')
FINAL_TEST_TARGET_START = pd.Period('2026-01', freq='M')
FINAL_TEST_TARGET_END = pd.Period('2026-07', freq='M')
best_ridge_setting = b15v2_tuning.iloc[0]
FINAL_RIDGE_WINDOW = int(best_ridge_setting['window'])
FINAL_RIDGE_ALPHA = float(best_ridge_setting['alpha'])
FINAL_RIDGE_BLEND = float(best_ridge_setting['보정비율'])
assert (FINAL_RIDGE_WINDOW, FINAL_RIDGE_ALPHA, FINAL_RIDGE_BLEND) == (3, 300.0, 0.30)

def _prepare_residual_frame(df):
    out = df.copy().reset_index(drop=True)
    out['b1_base'] = out['seasonal_naive_pred'] * out['growth_damped']
    out['target_resid_log'] = (
        np.log1p(out['target_passengers'])
        - np.log1p(out['b1_base'].clip(lower=0)))
    out['log_b1_base'] = np.log1p(out['b1_base'].clip(lower=0))
    out['log_seasonal_naive'] = np.log1p(
        out['seasonal_naive_pred'].clip(lower=0))
    for c in ['lag_1', 'lag_3', 'lag_12', 'ma_3', 'ma_12']:
        out[f'log_{c}'] = np.log1p(out[c].clip(lower=0))
    out['recent_vs_b1'] = out['log_ma_3'] - out['log_b1_base']
    return out

def _ridge_residual_score(train_source, score_source, origins,
                          window=3, alpha=300.0, blend=0.30):
    train_source = train_source.reset_index(drop=True)
    score_source = score_source.reset_index(drop=True)
    xnum_tr, xd_tr = _b15v2_design(train_source, feature_mode='core')
    xnum_sc, xd_sc = _b15v2_design(score_source, feature_mode='core')
    dummy_cols = xd_tr.columns.union(xd_sc.columns)
    xd_tr = xd_tr.reindex(columns=dummy_cols, fill_value=0)
    xd_sc = xd_sc.reindex(columns=dummy_cols, fill_value=0)
    parts = []
    for origin in origins:
        trmask = (
            (train_source['target_period'] <= origin)
            & (train_source['target_period'] >= origin - (window - 1)))
        for h in fl.HORIZONS:
            htr = trmask & (train_source['horizon'] == h)
            hte = ((score_source['origin'] == origin)
                   & (score_source['horizon'] == h))
            if not hte.any():
                continue
            scaler = StandardScaler().fit(xnum_tr.loc[htr])
            xtr = np.column_stack([
                scaler.transform(xnum_tr.loc[htr]), xd_tr.loc[htr].to_numpy()])
            xte = np.column_stack([
                scaler.transform(xnum_sc.loc[hte]), xd_sc.loc[hte].to_numpy()])
            model = Ridge(alpha=alpha).fit(
                xtr, train_source.loc[htr, 'target_resid_log'])
            part = score_source.loc[hte].copy()
            part['correction_full'] = model.predict(xte)
            parts.append(part)
    result = pd.concat(parts, ignore_index=True)
    result['pred_final'] = np.clip(np.expm1(
        result['log_b1_base'] + blend * result['correction_full']), 0, None)
    return result

# 학습 후보행은 각 Test 기준월까지 실제값이 확인된 최근 3개 대상월만 사용한다.
final_test_pool = fl.build_features(
    grids, universe, pd.period_range('2024-06', '2026-06', freq='M'))
final_test_pool = _prepare_residual_frame(final_test_pool)
final_test_score = final_test_pool[
    final_test_pool['origin'].isin(FINAL_TEST_ORIGINS)
    & final_test_pool['target_period'].between(
        FINAL_TEST_TARGET_START, FINAL_TEST_TARGET_END)
].copy()
final_test = _ridge_residual_score(
    final_test_pool, final_test_score, FINAL_TEST_ORIGINS,
    window=FINAL_RIDGE_WINDOW, alpha=FINAL_RIDGE_ALPHA,
    blend=FINAL_RIDGE_BLEND)

assert len(final_test) == 5915
assert final_test.groupby(['origin', 'horizon']).ngroups == 35
assert final_test['target_period'].min() == FINAL_TEST_TARGET_START
assert final_test['target_period'].max() == FINAL_TEST_TARGET_END
assert (final_test['max_feature_period'] <= final_test['origin']).all()

final_test_summary = pd.DataFrame([{
    'Fixed Test 평가 모델': FINAL_TEST_MODEL,
    '평가 행': len(final_test),
    'WAPE': _W(final_test, 'pred_final'),
    '순편향': ((final_test['pred_final'].sum()
             - final_test['target_passengers'].sum())
            / final_test['target_passengers'].sum()),
}])
display(final_test_summary.assign(
    WAPE=lambda d: d['WAPE'].map(lambda x: f'{x:.2%}'),
    순편향=lambda d: d['순편향'].map(lambda x: f'{x:+.2%}')))

final_test_by_h = final_test.groupby('horizon').apply(
    lambda x: _W(x, 'pred_final'), include_groups=False
).rename('WAPE')
print('Test 기준월:', FINAL_TEST_ORIGINS.min(), '–', FINAL_TEST_ORIGINS.max())
print('Test 대상월:', FINAL_TEST_TARGET_START, '–', FINAL_TEST_TARGET_END)
print('목적지×시점 행:', len(final_test),
      '| origin×horizon 조합:', final_test.groupby(['origin','horizon']).ngroups)
print('평가 모델 수: 1')
print('Fixed Test 평가 모델:', FINAL_TEST_MODEL)

assert np.isclose(final_test_summary.loc[0, 'WAPE'], 0.13316641934, atol=1e-7)


,Fixed Test 평가 모델,평가 행,WAPE,순편향
0,B1+Ridge 잔차보정,5915,13.32%,-3.82%


Test 기준월: 2025-08 – 2026-06
Test 대상월: 2026-01 – 2026-07
목적지×시점 행: 5915 | origin×horizon 조합: 35
평가 모델 수: 1
Fixed Test 평가 모델: B1+Ridge 잔차보정


In [36]:
# 선택 모델 Ridge의 Test WAPE를 예측거리별로 확인한다.
fig = go.Figure(go.Scatter(
    x=[f'h{h}' for h in final_test_by_h.index],
    y=final_test_by_h.values,
    mode='lines+markers+text',
    text=[f'{v:.2%}' for v in final_test_by_h.values],
    textposition='top center',
    line={'color': '#C7657A', 'width': 3}, marker={'size': 9},
    name='B1+Ridge 잔차보정',
))
fig.update_layout(
    title='선택 모델의 예측거리별 Fixed Test WAPE',
    xaxis_title='예측거리', yaxis_title='WAPE',
    template=PLOTLY_TEMPLATE, height=430,
    margin={'t': 75, 'r': 45, 'b': 60, 'l': 70},
    showlegend=False,
)
fig.update_yaxes(tickformat='.0%', range=[0, max(final_test_by_h) * 1.25])
fig.show(config={'displaylogo': False})

Ridge 잔차보정의 Fixed Test WAPE는 **13.32%** 다. Validation 11.47%보다 높아져 새 기간에서 오차가 커졌다. 이 Test 결과로 모델이나 파라미터를 다시 선택하지 않는다. 값은 여객 수요 예측의 오차이며 프로모션 집행 효과나 매출 증분을 검증한 값은 아니다.

## 7. 2026년 8–12월 수요 예측

2026년 7월을 기준으로 169개 목적지의 다음 5개월을 예측한다. Validation에서 선택한 Ridge 잔차보정의 설정을 그대로 사용한다. `추천월`은 이 5개월 중 예측 수요가 가장 높은 달이며, 과거 계절지수의 `성수월`이나 캠페인 시작일과는 다르다.

In [37]:
# 2026년 7월까지 알려진 자료로 Ridge 잔차보정 수요를 산출한다.
import tableau_utils as tu
from reconcile_forecast_candidates import summarize

deploy_origin = pd.Period('2026-07', freq='M')
deploy_score = fl.build_features(
    grids, universe, pd.period_range(deploy_origin, deploy_origin),
    require_target=False)
deploy_score = _prepare_residual_frame(deploy_score)
feat_deploy = _ridge_residual_score(
    final_test_pool, deploy_score, [deploy_origin],
    window=FINAL_RIDGE_WINDOW, alpha=FINAL_RIDGE_ALPHA,
    blend=FINAL_RIDGE_BLEND)
assert len(feat_deploy) == 845 and feat_deploy['dest'].nunique() == 169

# 참고구간은 Validation 상대잔차의 h별 경험 분위수다.
res = b15v2[(b15v2['pred_b15v2'] > 0)
            & (b15v2['target_passengers'] > 0)].assign(
    ratio=lambda d: d['target_passengers'] / d['pred_b15v2'])
qtab = res.groupby('horizon')['ratio'].quantile([.1, .9]).unstack()
qtab.columns = ['q10', 'q90']
feat_deploy = feat_deploy.merge(qtab, left_on='horizon', right_index=True)
feat_deploy['lo'] = feat_deploy['pred_final'] * feat_deploy['q10']
feat_deploy['hi'] = feat_deploy['pred_final'] * feat_deploy['q90']
feat_deploy['검증WAPE'] = feat_deploy['horizon'].map(b15v2_by_h).round(4)
feat_deploy['저신뢰'] = feat_deploy['target_month'].isin([11, 12])

annual_2025 = (panel[panel['period'].between(
    pd.Period('2025-01', 'M'), pd.Period('2025-12', 'M'))]
    .groupby('dest')['passengers'].sum().sort_values(ascending=False))
popularity_rank = annual_2025.rank(method='min', ascending=False)
di_deploy, pooled_deploy = fl.seasonal_index_asof(grids, deploy_origin)
peak_month = lambda dest: int(np.argmax(di_deploy.get(dest, pooled_deploy))) + 1

forecast_export = feat_deploy[[
    'origin', 'target_period', 'horizon', 'dest', 'seasonal_naive_pred',
    'pred_final', 'lo', 'hi', 'growth_damped', 'seasonal_index_target',
    'seasonal_index_own', '검증WAPE', '저신뢰'
]].copy()
forecast_export = tu.add_destination_labels(forecast_export, 'dest')
forecast_export['예측기준일'] = forecast_export['origin'].dt.to_timestamp()
forecast_export['예측월'] = forecast_export['target_period'].dt.to_timestamp()
forecast_export['연도'] = forecast_export['target_period'].dt.year
forecast_export['월'] = forecast_export['target_period'].dt.month
forecast_export['2025_인기순위'] = forecast_export['dest'].map(popularity_rank).astype('Int64')
forecast_export['표본구간'] = forecast_export['dest'].map(
    profile.set_index('destination_canonical_std')['sample_band'])
forecast_export['성수월'] = forecast_export['dest'].map(peak_month)
forecast_export['신뢰등급'] = np.where(forecast_export['저신뢰'], '저신뢰', '일반')
forecast_export['예상증가인원'] = (
    forecast_export['pred_final'] - forecast_export['seasonal_naive_pred'])
forecast_export['예상증가율'] = np.where(
    forecast_export['seasonal_naive_pred'] > 0,
    forecast_export['예상증가인원'] / forecast_export['seasonal_naive_pred'], np.nan)
forecast_export['월예측비중'] = forecast_export['pred_final'] / forecast_export.groupby(
    'target_period')['pred_final'].transform('sum')
forecast_export['실제수요'] = np.nan
forecast_export['절대오차'] = np.nan
forecast_export['APE'] = np.nan
forecast_export = forecast_export.rename(columns={
    'seasonal_naive_pred': '전년동월실제수요', 'pred_final': '예측수요',
    'lo': '예측하한', 'hi': '예측상한', 'growth_damped': '성장계수',
    'seasonal_index_target': '계절지수',
    'seasonal_index_own': '계절지수_자체산출여부'})
forecast_export = forecast_export[[
    '예측기준일', '예측월', '연도', '월', 'horizon', '목적지_DB', '목적지', '국가', '지역',
    '전년동월실제수요', '예측수요', '예측하한', '예측상한', '월예측비중',
    '예상증가인원', '예상증가율', '성장계수', '계절지수',
    '계절지수_자체산출여부', '검증WAPE', '신뢰등급', '2025_인기순위',
    '표본구간', '성수월', '실제수요', '절대오차', 'APE'
]].sort_values(['예측월', '예측수요'], ascending=[True, False])

ridge_candidates = summarize(forecast_export, FINAL_VALIDATION_MODEL)
popular_dests = set(ridge_candidates.loc[
    ridge_candidates['후보구분'] == '기존 인기', '목적지_DB'])
growth_dests = set(ridge_candidates.loc[
    ridge_candidates['후보구분'] == '성장 후보', '목적지_DB'])
forecast_export['후보구분'] = np.select(
    [forecast_export['목적지_DB'].isin(popular_dests),
     forecast_export['목적지_DB'].isin(growth_dests)],
    ['기존 인기', '성장 후보'], default='기타 모델링 목적지')
forecast_export.insert(22, '후보구분', forecast_export.pop('후보구분'))
assert len(ridge_candidates) == 55
assert ridge_candidates['후보구분'].value_counts().to_dict() == {
    '성장 후보': 35, '기존 인기': 20}
assert np.isclose(forecast_export['예측수요'].sum(), 20416998, atol=2)

print('목적지×월 예측:', len(forecast_export), '행 | 예측 합계:',
      f"{forecast_export['예측수요'].sum():,.0f}", '명')
print('검토 후보:', len(ridge_candidates), '곳')
display(forecast_export[['예측월', '목적지', '전년동월실제수요',
                         '예측수요', '성장계수']].head())

목적지×월 예측: 845 행 | 예측 합계: 20,416,998 명
검토 후보: 55 곳


,예측월,목적지,전년동월실제수요,예측수요,성장계수
0,2026-08-01,오사카 간사이,269618.0,322610.138482,1.195429
19,2026-08-01,도쿄 나리타,235700.0,279197.134285,1.191654
137,2026-08-01,후쿠오카,197851.0,216634.848574,1.115099
121,2026-08-01,상하이 푸둥,175526.0,207311.319738,1.185033
116,2026-08-01,타이베이,183408.0,194311.983038,1.094666


In [38]:
# 차트: 8~12월 월별 예측을 전년 동월 실적·참고구간과 함께 본다.
month_chart = feat_deploy.groupby('target_period').agg(
    전년실적=('seasonal_naive_pred', 'sum'), 예측=('pred_final', 'sum'),
    하한=('lo', 'sum'), 상한=('hi', 'sum')).reset_index()
month_chart['예측월'] = month_chart['target_period'].astype(str)

fig = go.Figure()
fig.add_trace(go.Scatter(x=month_chart['예측월'], y=month_chart['상한'],
                         mode='lines', line={'width': 0},
                         hoverinfo='skip', showlegend=False))
fig.add_trace(go.Scatter(x=month_chart['예측월'], y=month_chart['하한'],
                         name='참고구간 (q10~q90)', mode='lines',
                         line={'width': 0}, fill='tonexty',
                         fillcolor='rgba(79,134,181,0.18)'))
fig.add_trace(go.Scatter(x=month_chart['예측월'], y=month_chart['전년실적'],
                         name='2025년 실제 (전년 동월)', mode='lines+markers',
                         line={'color': '#9AA5B1', 'dash': 'dot'}))
fig.add_trace(go.Scatter(x=month_chart['예측월'], y=month_chart['예측'],
                         name='2026년 예측', mode='lines+markers+text',
                         line={'color': '#4F86B5', 'width': 3},
                         text=month_chart['예측'].map(lambda v: f'{v/1e4:,.0f}만'),
                         textposition='top center'))
fig.update_layout(
    title=('2026년 8~12월 월별 예측 수요와 참고구간'
           '<br><sup>기준월 2026-07 · 참고구간은 Validation 상대오차의 예측거리별 q10~q90</sup>'),
    xaxis_title='예측월', yaxis_title='여객 수', yaxis_tickformat=',',
    template=PLOTLY_TEMPLATE, height=470,
    legend={'orientation': 'h', 'y': -0.25})
fig.update_xaxes(type='category')
fig.show(config={'displaylogo': False})


169개 목적지×5개월의 Ridge 예측은 **845행**, 합계 **20,416,998명**이다. 참고구간은 2025년 Validation 잔차의 예측거리별 경험 분위수로, 확정적인 신뢰구간은 아니다. 11–12월은 기준월에서 4–5개월 떨어져 있으므로 새 실적이 들어오면 갱신한다. 이는 관측 수요의 예측치이며 캠페인에 따른 추가 예약을 뜻하지 않는다.

## 8. 프로모션 검토 후보

관측 구간에서는 누적 인기와 최근 증가 인원이 서로 다른 목적지를 가리켰고, 증가율만 보면 작은 시장을 과대평가할 수 있었다. 2026년 8–12월 예측에서 먼저 살펴볼 목적지를 **기존 인기 20곳과 성장 후보 35곳, 총 55곳**으로 선별한다. 55곳은 캠페인 집행 목록이 아니라 추가 검토를 위한 넓은 후보군이다.

- **기존 인기 20곳:** 2025년 연간 출발 여객 상위 20곳이다. 2026년 8–12월 예측이 전년 같은 기간보다 늘면 `인기`, 줄면 `방어`로 표시한다.
- **성장 후보 35곳:** 인기 20곳 밖에서 24개월 이상 관측되고, 2025년 8–12월 실적 3만 명 이상, 2026년 같은 기간 예측 5만 명 이상, 성장계수 1 초과, 예상 증가 인원 양수, 증가 예상월 3개월 이상인 목적지다. 이 중 타이중·히로시마·시즈오카는 작은 시장의 실험 가능성을 따로 살펴보기 위해 `탐색`으로 표시했고, 나머지는 `성장`으로 표시했다. 탐색 3곳은 자동 순위나 별도의 통계적 기준으로 뽑은 집합이 아니다.

| 검토 트랙 | 뜻 | 먼저 확인할 것 |
|---|---|---|
| 인기 | 인기 20곳 중 예측 수요가 전년보다 증가 | 큰 수요에 맞는 상품·좌석 재고와 수익성 |
| 성장 | 성장 조건을 통과했고 탐색으로 별도 표시하지 않은 목적지 | 증가 **인원**의 크기와 월별 지속성 |
| 탐색 | 성장 조건 통과 목적지 중 별도 표시한 3곳 | 제한된 실험에 필요한 규모·표본·상품성 |
| 방어 | 인기 20곳 중 예측 수요가 전년보다 감소 | 가격·공급·예약 흐름 등 감소 원인 |

3만 명·5만 명·3개월은 통계적으로 추정한 경계가 아니라 작은 시장의 높은 증가율만으로 후보를 고르지 않기 위한 분석상 조건이다. `성수월`은 과거 계절지수의 최대월이고, `추천월`은 2026년 8–12월 **예측 수요 최대월**이다. 추천월은 캠페인 시작 시점이나 최대 수익월을 뜻하지 않는다.


In [39]:
# 넓은 후보군을 트랙별로 확인하고, 먼저 살펴볼 대상을 정렬한다.
track_order = ['인기', '성장', '탐색', '방어']
track_sort = {
    '인기': ('예측수요', False),
    '성장': ('예상증가인원', False),
    '탐색': ('예상증가인원', False),
    '방어': ('예상증가인원', True),
}
ordered_by_track = {}
for track in track_order:
    metric, ascending = track_sort[track]
    ordered_by_track[track] = (ridge_candidates.loc[
        ridge_candidates['검토트랙'].eq(track)]
        .sort_values([metric, '목적지_DB'], ascending=[ascending, True])
        .reset_index(drop=True))

track_counts = pd.DataFrame({
    '검토트랙': track_order,
    '목적지 수': [len(ordered_by_track[t]) for t in track_order],
})
assert track_counts['목적지 수'].tolist() == [11, 32, 3, 9]
assert track_counts['목적지 수'].sum() == 55
print('1차 검토 후보: 기존 인기 20곳 + 성장 후보 35곳 = 55곳')
display(track_counts)

# 각 트랙에서 최대 5곳을 펼친다. 탐색은 3곳 모두 표시한다.
first_review = pd.concat(
    [ordered_by_track[t].head(5).assign(
        검토순서=lambda d: np.arange(1, len(d) + 1))
     for t in track_order], ignore_index=True)
assert first_review['검토트랙'].value_counts().reindex(track_order).tolist() == [5, 5, 3, 5]
first_review = first_review[['검토트랙', '검토순서', '목적지', '전년수요',
                             '예측수요', '예상증가인원', '추천월']].copy()
for col in ['전년수요', '예측수요', '예상증가인원']:
    first_review[col] = first_review[col].map(lambda x: f'{x:+,.0f}' if col == '예상증가인원' else f'{x:,.0f}')
first_review['추천월'] = first_review['추천월'].map(lambda x: f'{int(x)}월')
display(first_review)


1차 검토 후보: 기존 인기 20곳 + 성장 후보 35곳 = 55곳


,검토트랙,목적지 수
0,인기,11
1,성장,32
2,탐색,3
3,방어,9


,검토트랙,검토순서,목적지,전년수요,예측수요,예상증가인원,추천월
0,인기,1,오사카 간사이,"1,580,068","1,854,611","+274,543",12월
1,인기,2,도쿄 나리타,"1,234,043","1,449,787","+215,744",12월
2,인기,3,후쿠오카,"1,053,325","1,140,694","+87,369",12월
3,인기,4,타이베이,"942,664","993,205","+50,541",12월
4,인기,5,상하이 푸둥,"825,929","974,788","+148,859",8월
5,성장,1,가오슝,"199,517","250,427","+50,910",12월
6,성장,2,지난,"70,138","108,181","+38,043",10월
7,성장,3,광저우,"184,661","214,951","+30,290",10월
8,성장,4,다롄,"118,530","148,768","+30,238",8월
9,성장,5,시애틀,"123,676","147,010","+23,334",8월


### 55곳 전체 목적지

아래 이름은 같은 후보군을 검토 트랙별로 모두 펼친 것이다. `인기`는 예측 수요가 큰 순서, `성장`과 `탐색`은 예상 증가 인원이 큰 순서, `방어`는 예상 감소 인원이 큰 순서다. 순서는 캠페인 수익률 순위가 아니다.


In [40]:
# 55곳을 생략 없이 표시한다. 긴 성장 목록은 여덟 곳씩 나누어 읽는다.
from IPython.display import Markdown
for track in track_order:
    names = ordered_by_track[track]['목적지'].tolist()
    lines = [', '.join(names[i:i + 8]) for i in range(0, len(names), 8)]
    display(Markdown(f"**{track} ({len(names)}곳)**  \n" + "  \n".join(lines)))
assert sum(len(ordered_by_track[t]) for t in track_order) == 55


**인기 (11곳)**  
오사카 간사이, 도쿄 나리타, 후쿠오카, 타이베이, 상하이 푸둥, 홍콩, 도쿄 하네다, 삿포로  
베이징, 나고야, 오키나와

**성장 (32곳)**  
가오슝, 지난, 광저우, 다롄, 시애틀, 런던 히드로, 밴쿠버, 애틀랜타  
자카르타, 항저우, 홍차오, 난징, 웨이하이, 워싱턴덜레스, 다카마쓰, 옌타이  
톈진, 마즈야마, 하얼빈, 옌지, 텐푸, 뉴욕JFK, 다싱, 장가계  
샌프란시스코, 울란바토르, 시안, 선양, 구마모토, 토론토, 장춘, 달라스

**탐색 (3곳)**  
타이중, 히로시마, 시즈오카

**방어 (9곳)**  
다낭, 방콕, 나트랑 깜라인, 마닐라, 싱가포르, 하노이, 로스앤젤레스, 호치민  
칭다오

### 검토 결론

**집행 가능성을 먼저 확인할 목적지는 10곳**이다. 수요 규모가 큰 인기 트랙의 **오사카 간사이·도쿄 나리타·후쿠오카·타이베이·상하이 푸둥**, 예상 증가 인원이 큰 성장 트랙의 **가오슝·지난·광저우·다롄·시애틀**에서 상품·좌석 재고와 수익성을 비교한다. 각 트랙의 상위 5곳은 검토 범위를 정한 것이며, 실제 집행 대상과 예산은 사업 지표를 확인한 뒤 결정한다.

**타이중·히로시마·시즈오카**는 탐색 트랙 3곳 모두를 소규모 실험 적합성 검토에 포함한다. 방어 트랙의 **다낭·방콕·나트랑 깜라인·마닐라·싱가포르**는 예상 감소 인원이 큰 순서로 가격·공급·예약 흐름을 조사한다. 이렇게 먼저 살펴볼 18곳 외의 목적지도 55곳 전체 목록에 두고, 선행 지표와 사업 조건이 달라질 때 검토 순서를 갱신한다.

예측 수요는 목적지의 향후 여객 규모를 가늠하는 값이다. 집행 여부와 시점을 정할 때는 가격·할인 반응, 좌석·운항 공급, 검색·예약 선행 지표와 리드타임, 마진·예산을 함께 확인한다. 캠페인의 성과는 이후 순증 예약과 이익으로 평가한다. 실제 8–12월 수요가 쌓이면 월별 WAPE·순편향·후보 순위 안정성을 추적한다.

모델 대상 169개는 2026년 7월 프로필에서 고정했다. 과거 기준월마다 목적지 적격성을 다시 선택하는 백테스트는 후속 검증 범위다.